In [ ]:
import ee
import geemap 
import os
from bin import functions
ee.Authenticate()

# Set GEE_PROJECT if your Earth Engine cloud project is required.
project = os.environ.get("GEE_PROJECT")
if project:
    ee.Initialize(project=project)
else:
    ee.Initialize()

# Principal paths 
### 1) this code could donwload images from google earth engine and prosprocesing in local 
### 2) using shapes draw myself for limits and cut parts aren't interesting 
### 3) All shapes are in formats geojson 


In [ ]:
raw_path = os.getcwd()
water_bodie_path= raw_path + r'\Download\ligua_petorca_sentinel_TOA'
Map = geemap.Map(center=[-32.405493, -71.404779],zoom=14,basemap = "Esri.WorldImagery")
Map


In [ ]:
path_line = raw_path + r'\shapes\ligua_petorca_coast\geometry_line_coast.geojson'
path_geometry_coast = raw_path + r'\shapes\ligua_petorca_coast\geometry_coast.geojson'
print(path_geometry_coast)

In [ ]:
path_geometry= raw_path + r'\shapes\ligua_petorca_coast\geometry.geojson'
geometry = geemap.geojson_to_ee(path_geometry)
geometry = geometry.geometry()
print(type(geometry))

In [ ]:
Map.centerObject(geometry, 14) 
Map.addLayer(geometry, {}, 'Imported polygoon')
Map

# Download sentinel NIR, RGB y NDWI, if you downloaded, ingnoring this step

In [ ]:
functions.download_sentinel(water_bodie_path,geometry)

## If you have already download satelital image, folow this step 

In [ ]:
functions.mask_nir(water_bodie_path)
# Mask using otsu method in band NIR and dilated mask 1 pixel each direction

In [ ]:
functions.cut_coast(raw_path,water_bodie_path,path_geometry_coast)
# cut coast used shapes 

In [ ]:
functions.extract_ndwi2(raw_path,water_bodie_path)
#extraect NDWI by mask tif 

# CLASIFICATION PIXEL 

## Calculate histogram
## 1) LL: land level 
## 2) W: water level
### Method used by Cooley et al. (2017)

In [ ]:

path_ndwi_wetland = water_bodie_path + r'\ndwi_wetland'
files_in_directory = os.listdir(path_ndwi_wetland)
files_tif = [file for file in files_in_directory if file.lower().endswith('.tif')]
results = functions.land_water_level(path_ndwi_wetland,files_tif)

ndwi, name_image = results

## Clasification

In [ ]:
import rasterio
import numpy as np
for i in range(len(name_image)):
    image = os.path.join(path_ndwi_wetland, name_image[i])
    name = name_image[i]
    
    with rasterio.open(image) as src:
        modified_raster_array = src.read(1)  # Read the first band (assumes single-band raster)
        profile = src.profile

    LL = ndwi[0, 0]  # Lower threshold
    WL = ndwi[0, 1]  # Upper threshold

    # Masks for low and wetland values
    mask_LL = modified_raster_array < LL
    mask_WL = modified_raster_array >= WL

    # Initialize the result array
    result = 100 * (modified_raster_array - LL) / (WL - LL)
    
    # Set NaN where mask_LL is True
    result = np.where(mask_LL, np.nan, result)

    # Set 100 where mask_WL is True
    #result[mask_WL] = 100

    # Adjust profile for NaN handling (if necessary, you can set a NoData value)
    profile.update(dtype=rasterio.float32, nodata=np.nan)

    # Prepare the output path
    final_name = 'wetland_classification' + name
    dir_final = os.path.join(water_bodie_path, 'classification', final_name)

    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(dir_final), exist_ok=True)

    # Write the result to the output file
    with rasterio.open(dir_final, 'w', **profile) as dst:
        dst.write(result.astype(rasterio.float32), 1)  # Ensure correct dtype when writing